In [1]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pyarrow as pa
import pyarrow.parquet as pq

KT1_DATASET = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_merged")
QUESTIONS_FILE = Path(r"D:\Lokesh files\Projects\LearningDNA\contents\questions.csv")
OUTPUT_DIR = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed")

NUM_THREADS = 6

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("LEARNINGDNA - KT1 FEATURE PROCESSING")
print("=" * 60)
print(f"Input:    {KT1_DATASET}")
print(f"Questions: {QUESTIONS_FILE}")
print(f"Output:   {OUTPUT_DIR}")
print(f"Threads:  {NUM_THREADS}")

LEARNINGDNA - KT1 FEATURE PROCESSING
Input:    D:\Lokesh files\Projects\LearningDNA\processed\kt1_merged
Questions: D:\Lokesh files\Projects\LearningDNA\contents\questions.csv
Output:   D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed
Threads:  6


In [2]:
parts = sorted(KT1_DATASET.glob("part_*.parquet"))

print("Parquet parts found:", f"{len(parts):,}")

if not parts:
    raise FileNotFoundError(f"No Parquet parts found in {KT1_DATASET}")

sample = pd.read_parquet(parts[0])

print("\nSample shape:", sample.shape)
print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 rows:")
display(sample.head())

Parquet parts found: 15,687

Sample shape: (41088, 6)

Columns:
['user_id', 'timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time']

Data types:
user_id                  int64
timestamp                Int64
solving_id               Int64
question_id     string[python]
user_answer     string[python]
elapsed_time             Int64
dtype: object

First 5 rows:


,user_id,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1,1565096190868,1,q5012,b,38000
1,1,1565096221062,2,q4706,c,24000
2,1,1565096293432,3,q4366,b,68000
3,1,1565096339668,4,q4829,a,42000
4,1,1565096401774,5,q6528,b,59000


In [3]:
print("Loading question metadata...")

questions_df = pd.read_csv(
    QUESTIONS_FILE,
    dtype={
        "question_id": "string",
        "correct_answer": "string",
        "tags": "string"
    }
)

questions_df = questions_df[
    ["question_id", "correct_answer", "tags"]
].copy()

print("Questions loaded:", f"{len(questions_df):,}")
print("Duplicate question IDs:", questions_df["question_id"].duplicated().sum())

print("\nFirst 5 questions:")
display(questions_df.head())

Loading question metadata...
Questions loaded: 13,169
Duplicate question IDs: 0

First 5 questions:


,question_id,correct_answer,tags
0,q1,b,1;2;179;181
1,q2,a,15;2;182
2,q3,b,14;2;179;183
3,q4,b,9;2;179;184
4,q5,c,8;2;179;181


In [4]:
def process_part(input_file, output_file):
    try:
        df = pd.read_parquet(input_file)

        df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce").astype("int64")
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("int64")
        df["solving_id"] = pd.to_numeric(df["solving_id"], errors="coerce").astype("int64")
        df["elapsed_time"] = pd.to_numeric(df["elapsed_time"], errors="coerce").astype("int64")
        df["question_id"] = df["question_id"].astype("string")
        df["user_answer"] = df["user_answer"].astype("string")

        df = df.merge(
            questions_df,
            on="question_id",
            how="left",
            validate="many_to_one"
        )

        df["answer_missing"] = df["user_answer"].isna().astype("int8")

        df["correctness"] = (
            df["user_answer"].notna() &
            df["correct_answer"].notna() &
            (df["user_answer"] == df["correct_answer"])
        ).astype("int8")

        df = df.sort_values(
            ["user_id", "timestamp", "solving_id"]
        ).reset_index(drop=True)

        df["attempt_index"] = (
            df.groupby(
                ["user_id", "question_id"],
                sort=False
            ).cumcount() + 1
        ).astype("int32")

        df["time_delta_seconds"] = (
            df.groupby("user_id", sort=False)["timestamp"]
            .diff()
            .div(1000.0)
            .fillna(0.0)
        )

        df["elapsed_seconds"] = (
            df["elapsed_time"] / 1000.0
        )

        df["log_elapsed_time"] = np.log1p(
            df["elapsed_seconds"].clip(lower=0)
        )

        df["log_time_delta"] = np.log1p(
            df["time_delta_seconds"].clip(lower=0)
        )

        df = df[
            [
                "user_id",
                "timestamp",
                "solving_id",
                "question_id",
                "user_answer",
                "correct_answer",
                "correctness",
                "answer_missing",
                "tags",
                "elapsed_time",
                "elapsed_seconds",
                "attempt_index",
                "time_delta_seconds",
                "log_elapsed_time",
                "log_time_delta"
            ]
        ]

        df.to_parquet(
            output_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        return {
            "file": input_file.name,
            "rows": len(df),
            "status": "success",
            "error": None
        }

    except Exception as e:
        return {
            "file": input_file.name,
            "rows": 0,
            "status": "failed",
            "error": str(e)
        }

In [5]:
TEST_DIR = OUTPUT_DIR / "test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

test_parts = parts[:3]

print("=" * 60)
print("TESTING FEATURE PROCESSING")
print("=" * 60)

for part in test_parts:
    output_file = TEST_DIR / part.name
    result = process_part(part, output_file)
    print(result)

print("\nTest output:")
test_df = pd.read_parquet(TEST_DIR / test_parts[0].name)

print("Shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 10 rows:")
display(test_df.head(10))

print("\nData types:")
print(test_df.dtypes)

TESTING FEATURE PROCESSING
{'file': 'part_000000.parquet', 'rows': 41088, 'status': 'success', 'error': None}
{'file': 'part_000001.parquet', 'rows': 37524, 'status': 'success', 'error': None}
{'file': 'part_000002.parquet', 'rows': 51749, 'status': 'success', 'error': None}

Test output:
Shape: (41088, 15)

Columns:
['user_id', 'timestamp', 'solving_id', 'question_id', 'user_answer', 'correct_answer', 'correctness', 'answer_missing', 'tags', 'elapsed_time', 'elapsed_seconds', 'attempt_index', 'time_delta_seconds', 'log_elapsed_time', 'log_time_delta']

First 10 rows:


,user_id,timestamp,solving_id,question_id,user_answer,correct_answer,correctness,answer_missing,tags,elapsed_time,elapsed_seconds,attempt_index,time_delta_seconds,log_elapsed_time,log_time_delta
0,1,1565096190868,1,q5012,b,c,0,0,74,38000,38.0,1,0.000,3.663562,0.000000
1,1,1565096221062,2,q4706,c,c,1,0,71,24000,24.0,1,30.194,3.218876,3.440226
2,1,1565096293432,3,q4366,b,b,1,0,103,68000,68.0,1,72.370,4.234107,4.295515
3,1,1565096339668,4,q4829,a,c,0,0,83,42000,42.0,1,46.236,3.761200,3.855156
4,1,1565096401774,5,q6528,b,d,0,0,90,59000,59.0,1,62.106,4.094345,4.144816
5,1,1565096463370,6,q4793,a,b,0,0,82,58000,58.0,1,61.596,4.077537,4.136701
6,1,1565096501746,7,q6488,a,a,1,0,77,35000,35.0,1,38.376,3.583519,3.673156
7,1,1565097101361,8,q356,b,b,1,0,30;34;42;181;183,23000,23.0,1,599.615,3.178054,6.397954
8,1,1565097171393,9,q1382,c,c,1,0,24;43;184;181,22000,22.0,1,70.032,3.135494,4.263130
9,1,1565097240758,10,q830,b,b,1,0,27;28;45;184;182,25000,25.0,1,69.365,3.258097,4.253696



Data types:
user_id                        int64
timestamp                      int64
solving_id                     int64
question_id           string[python]
user_answer           string[python]
correct_answer        string[python]
correctness                     int8
answer_missing                  int8
tags                  string[python]
elapsed_time                   int64
elapsed_seconds              float64
attempt_index                  int32
time_delta_seconds           float64
log_elapsed_time             float64
log_time_delta               float64
dtype: object


In [6]:
print("=" * 60)
print("FEATURE VALIDATION")
print("=" * 60)

print("Missing question metadata:",
      test_df["correct_answer"].isna().sum())

print("Missing answers:",
      test_df["answer_missing"].sum())

print("Correct answers:",
      (test_df["correctness"] == 1).sum())

print("Incorrect answers:",
      (
          (test_df["correctness"] == 0) &
          (test_df["answer_missing"] == 0)
      ).sum())

print("Negative elapsed:",
      (test_df["elapsed_seconds"] < 0).sum())

print("Negative time delta:",
      (test_df["time_delta_seconds"] < 0).sum())

print("Minimum attempt index:",
      test_df["attempt_index"].min())

print("Maximum attempt index:",
      test_df["attempt_index"].max())

print("\nTemporal feature statistics:")
display(
    test_df[
        [
            "elapsed_seconds",
            "time_delta_seconds",
            "log_elapsed_time",
            "log_time_delta"
        ]
    ].describe()
)

FEATURE VALIDATION
Missing question metadata: 0
Missing answers: 1036
Correct answers: 27763
Incorrect answers: 12289
Negative elapsed: 0
Negative time delta: 0
Minimum attempt index: 1
Maximum attempt index: 22

Temporal feature statistics:


,elapsed_seconds,time_delta_seconds,log_elapsed_time,log_time_delta
count,41088.000000,4.108800e+04,41088.000000,41088.000000
mean,28.857186,8.664354e+03,3.045925,2.809205
std,96.561366,2.204964e+05,0.840059,2.731093
min,0.000000,0.000000e+00,0.000000,0.000000
25%,16.000000,2.700000e-02,2.833213,0.026642
50%,23.333000,2.700000e+01,3.191833,3.332205
75%,30.000000,8.157350e+01,3.433987,4.413689
max,9049.000000,2.688496e+07,9.110520,17.107078


In [7]:
print("=" * 60)
print("STARTING FULL KT1 FEATURE PROCESSING")
print("=" * 60)

start = time.time()
completed = 0
total_rows = 0
failed = []

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {}

    for part in parts:
        output_file = OUTPUT_DIR / part.name
        futures[executor.submit(process_part, part, output_file)] = part.name

    for future in as_completed(futures):
        result = future.result()
        completed += 1
        total_rows += result["rows"]

        if result["status"] == "failed":
            failed.append(result)

        if completed % 50 == 0 or completed == len(parts):
            elapsed = time.time() - start
            percent = completed / len(parts) * 100

            print(
                f"[{completed:,}/{len(parts):,}] "
                f"{percent:6.2f}% | "
                f"Rows: {total_rows:,} | "
                f"Failed: {len(failed):,} | "
                f"Time: {elapsed / 60:.1f} min"
            )

print("\n" + "=" * 60)
print("FULL PROCESSING COMPLETE")
print("=" * 60)
print(f"Parts processed: {completed:,}")
print(f"Rows processed: {total_rows:,}")
print(f"Failed parts:   {len(failed):,}")
print(f"Time taken:     {(time.time() - start) / 60:.2f} min")

if failed:
    with open(OUTPUT_DIR / "failed_parts.txt", "w", encoding="utf-8") as f:
        for item in failed:
            f.write(f"{item['file']}\t{item['error']}\n")

STARTING FULL KT1 FEATURE PROCESSING
[50/15,687]   0.32% | Rows: 3,514,237 | Failed: 0 | Time: 0.5 min
[100/15,687]   0.64% | Rows: 6,225,953 | Failed: 0 | Time: 0.9 min
[150/15,687]   0.96% | Rows: 8,778,379 | Failed: 0 | Time: 1.2 min
[200/15,687]   1.27% | Rows: 12,124,427 | Failed: 0 | Time: 1.7 min
[250/15,687]   1.59% | Rows: 14,687,454 | Failed: 0 | Time: 2.0 min
[300/15,687]   1.91% | Rows: 16,461,993 | Failed: 0 | Time: 2.3 min
[350/15,687]   2.23% | Rows: 17,770,527 | Failed: 0 | Time: 2.6 min
[400/15,687]   2.55% | Rows: 18,813,758 | Failed: 0 | Time: 2.8 min
[450/15,687]   2.87% | Rows: 20,344,674 | Failed: 0 | Time: 3.0 min
[500/15,687]   3.19% | Rows: 21,455,968 | Failed: 0 | Time: 3.2 min
[550/15,687]   3.51% | Rows: 22,301,344 | Failed: 0 | Time: 3.4 min
[600/15,687]   3.82% | Rows: 23,076,225 | Failed: 0 | Time: 3.6 min
[650/15,687]   4.14% | Rows: 23,766,088 | Failed: 0 | Time: 3.8 min
[700/15,687]   4.46% | Rows: 24,391,501 | Failed: 0 | Time: 3.9 min
[750/15,687]   

In [8]:
processed_parts = sorted(OUTPUT_DIR.glob("part_*.parquet"))

print("=" * 60)
print("PROCESSED DATASET CHECK")
print("=" * 60)

print("Processed parts:", f"{len(processed_parts):,}")

if len(processed_parts) != len(parts):
    print(
        f"WARNING: Expected {len(parts):,} parts, "
        f"found {len(processed_parts):,}."
    )
else:
    print("Part count: PASS")

sample = pd.read_parquet(processed_parts[0])

print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nSample:")
display(sample.head(10))

PROCESSED DATASET CHECK
Processed parts: 15,687
Part count: PASS

Columns:
['user_id', 'timestamp', 'solving_id', 'question_id', 'user_answer', 'correct_answer', 'correctness', 'answer_missing', 'tags', 'elapsed_time', 'elapsed_seconds', 'attempt_index', 'time_delta_seconds', 'log_elapsed_time', 'log_time_delta']

Data types:
user_id                        int64
timestamp                      int64
solving_id                     int64
question_id           string[python]
user_answer           string[python]
correct_answer        string[python]
correctness                     int8
answer_missing                  int8
tags                  string[python]
elapsed_time                   int64
elapsed_seconds              float64
attempt_index                  int32
time_delta_seconds           float64
log_elapsed_time             float64
log_time_delta               float64
dtype: object

Sample:


,user_id,timestamp,solving_id,question_id,user_answer,correct_answer,correctness,answer_missing,tags,elapsed_time,elapsed_seconds,attempt_index,time_delta_seconds,log_elapsed_time,log_time_delta
0,1,1565096190868,1,q5012,b,c,0,0,74,38000,38.0,1,0.000,3.663562,0.000000
1,1,1565096221062,2,q4706,c,c,1,0,71,24000,24.0,1,30.194,3.218876,3.440226
2,1,1565096293432,3,q4366,b,b,1,0,103,68000,68.0,1,72.370,4.234107,4.295515
3,1,1565096339668,4,q4829,a,c,0,0,83,42000,42.0,1,46.236,3.761200,3.855156
4,1,1565096401774,5,q6528,b,d,0,0,90,59000,59.0,1,62.106,4.094345,4.144816
5,1,1565096463370,6,q4793,a,b,0,0,82,58000,58.0,1,61.596,4.077537,4.136701
6,1,1565096501746,7,q6488,a,a,1,0,77,35000,35.0,1,38.376,3.583519,3.673156
7,1,1565097101361,8,q356,b,b,1,0,30;34;42;181;183,23000,23.0,1,599.615,3.178054,6.397954
8,1,1565097171393,9,q1382,c,c,1,0,24;43;184;181,22000,22.0,1,70.032,3.135494,4.263130
9,1,1565097240758,10,q830,b,b,1,0,27;28;45;184;182,25000,25.0,1,69.365,3.258097,4.253696


In [9]:
print("=" * 60)
print("FULL DATASET STATISTICS")
print("=" * 60)

total_rows = 0
total_missing_answers = 0
total_correct = 0
total_incorrect = 0
total_duplicate_rows = 0
total_negative_elapsed = 0
total_negative_delta = 0

min_user = None
max_user = None

unique_users = set()

start = time.time()

for i, part in enumerate(processed_parts, 1):
    df = pd.read_parquet(part)

    total_rows += len(df)

    total_missing_answers += int(
        df["answer_missing"].sum()
    )

    total_correct += int(
        df["correctness"].sum()
    )

    total_incorrect += int(
        (
            (df["correctness"] == 0) &
            (df["answer_missing"] == 0)
        ).sum()
    )

    total_duplicate_rows += int(
        df.duplicated().sum()
    )

    total_negative_elapsed += int(
        (df["elapsed_seconds"] < 0).sum()
    )

    total_negative_delta += int(
        (df["time_delta_seconds"] < 0).sum()
    )

    unique_users.update(
        df["user_id"].unique()
    )

    part_min = int(df["user_id"].min())
    part_max = int(df["user_id"].max())

    min_user = part_min if min_user is None else min(min_user, part_min)
    max_user = part_max if max_user is None else max(max_user, part_max)

    if i % 500 == 0 or i == len(processed_parts):
        print(
            f"[{i:,}/{len(processed_parts):,}] "
            f"Rows: {total_rows:,} | "
            f"Time: {(time.time() - start) / 60:.1f} min"
        )

print("\n" + "=" * 60)
print("FINAL STATISTICS")
print("=" * 60)

print(f"Total interactions : {total_rows:,}")
print(f"Unique students   : {len(unique_users):,}")
print(f"Minimum user ID   : {min_user}")
print(f"Maximum user ID   : {max_user}")
print(f"Missing answers   : {total_missing_answers:,}")
print(f"Correct answers   : {total_correct:,}")
print(f"Incorrect answers : {total_incorrect:,}")
print(f"Duplicate rows    : {total_duplicate_rows:,}")
print(f"Negative elapsed  : {total_negative_elapsed:,}")
print(f"Negative delta    : {total_negative_delta:,}")

FULL DATASET STATISTICS
[500/15,687] Rows: 21,455,968 | Time: 0.6 min
[1,000/15,687] Rows: 27,785,205 | Time: 0.9 min
[1,500/15,687] Rows: 31,805,157 | Time: 1.2 min
[2,000/15,687] Rows: 36,511,370 | Time: 1.5 min
[2,500/15,687] Rows: 40,426,996 | Time: 1.8 min
[3,000/15,687] Rows: 45,333,514 | Time: 2.1 min
[3,500/15,687] Rows: 50,151,625 | Time: 2.4 min
[4,000/15,687] Rows: 53,442,274 | Time: 2.7 min
[4,500/15,687] Rows: 56,814,919 | Time: 3.0 min
[5,000/15,687] Rows: 60,931,258 | Time: 3.3 min
[5,500/15,687] Rows: 63,321,161 | Time: 3.6 min
[6,000/15,687] Rows: 64,893,773 | Time: 3.8 min
[6,500/15,687] Rows: 66,115,791 | Time: 4.0 min
[7,000/15,687] Rows: 67,713,234 | Time: 4.3 min
[7,500/15,687] Rows: 69,577,024 | Time: 4.5 min
[8,000/15,687] Rows: 71,473,834 | Time: 4.8 min
[8,500/15,687] Rows: 73,223,695 | Time: 5.0 min
[9,000/15,687] Rows: 74,739,866 | Time: 5.2 min
[9,500/15,687] Rows: 76,270,452 | Time: 5.5 min
[10,000/15,687] Rows: 77,899,727 | Time: 5.7 min
[10,500/15,687] R

In [10]:
TEST_USER_ID = 1

student_parts = []

for part in processed_parts:
    df = pd.read_parquet(
        part,
        filters=[("user_id", "==", TEST_USER_ID)]
    )

    if not df.empty:
        student_parts.append(df)

if student_parts:
    student = pd.concat(student_parts, ignore_index=True)
    student = student.sort_values(
        ["timestamp", "solving_id"]
    ).reset_index(drop=True)

    print("=" * 60)
    print(f"STUDENT {TEST_USER_ID}")
    print("=" * 60)

    print("Interactions:", len(student))
    print("Questions:", student["question_id"].nunique())
    print("Correct:", student["correctness"].sum())
    print("Accuracy:", student["correctness"].mean())

    display(
        student[
            [
                "user_id",
                "timestamp",
                "question_id",
                "user_answer",
                "correct_answer",
                "correctness",
                "attempt_index",
                "elapsed_seconds",
                "time_delta_seconds",
                "log_elapsed_time",
                "log_time_delta"
            ]
        ].head(20)
    )
else:
    print(f"User {TEST_USER_ID} was not found.")

STUDENT 1
Interactions: 1082
Questions: 926
Correct: 753
Accuracy: 0.6959334565619224


,user_id,timestamp,question_id,user_answer,correct_answer,correctness,attempt_index,elapsed_seconds,time_delta_seconds,log_elapsed_time,log_time_delta
0,1,1565096190868,q5012,b,c,0,1,38.0,0.000,3.663562,0.000000
1,1,1565096221062,q4706,c,c,1,1,24.0,30.194,3.218876,3.440226
2,1,1565096293432,q4366,b,b,1,1,68.0,72.370,4.234107,4.295515
3,1,1565096339668,q4829,a,c,0,1,42.0,46.236,3.761200,3.855156
4,1,1565096401774,q6528,b,d,0,1,59.0,62.106,4.094345,4.144816
5,1,1565096463370,q4793,a,b,0,1,58.0,61.596,4.077537,4.136701
6,1,1565096501746,q6488,a,a,1,1,35.0,38.376,3.583519,3.673156
7,1,1565097101361,q356,b,b,1,1,23.0,599.615,3.178054,6.397954
8,1,1565097171393,q1382,c,c,1,1,22.0,70.032,3.135494,4.263130
9,1,1565097240758,q830,b,b,1,1,25.0,69.365,3.258097,4.253696
